In [ ]:
# Imports and environment first, all in one place

from dotenv import load_dotenv
from IPython.display import Image, display
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langchain_mcp_adapters.client import MultiServerMCPClient
from urllib.parse import urlparse
from langchain.messages import ToolMessage

load_dotenv(override=True)


In [ ]:
client = MultiServerMCPClient(
    {
        "playwright": {
            "transport": "stdio",
            "command": "npx",
            "args": ["-y", "@playwright/mcp@latest", "--isolated"],
        }
    }
)

browser_tools = await client.get_tools()
print(f"Loaded {len(browser_tools)} browser tools:")
for t in browser_tools:
    print(" -", t.name)

# we could have written all these 23 tools ourselves, each with a @tool decorator, and a natural language description of what we want to do, and a description of each of the parameters, and a call to node in playwright to do the actual work.


### Middleware


In [ ]:
BLOCKED_WEBSITES = {
    "facebook.com",
    "instagram.com",
    "tiktok.com",
    "news.ycombinator.com",
}


# IMPORTANT: if ainvoke is used, then the middleware must also be async function.
@wrap_tool_call
async def block_websites(request, handler):

    tool_name = request.tool_call["name"]
    args = request.tool_call["args"]

    # Only check tools that contain a URL (for llm to go to a website, it needs to use this tool, and this tool have url in its args which we can check against the blocked list)
    if tool_name == "browser_navigate":
        url = args.get("url")

        if url:
            hostname = urlparse(
                url
            ).hostname  # turns url into an object to get the host name

            if hostname in BLOCKED_WEBSITES:
                return ToolMessage(
                    content=(
                        f"{hostname} is blocked. "
                        "Do not visit this website. Direct user to use another source."
                    ),
                    tool_call_id=request.tool_call["id"],
                )

    # Website is OK -> execute Playwright normally
    return await handler(
        request
    )  # make sure to have await here otherwise we are just returning a coroutine.


In [ ]:
browser_agent = create_agent(
    model="openai:gpt-5.5",
    tools=browser_tools,
    system_prompt="You are a web research assistant. Use the browser tools to complete the task, then report clearly.",
    checkpointer=MemorySaver(),
    middleware=[block_websites],
)

config = {"configurable": {"thread_id": "fetch-y-combinator-news"}}
await browser_agent.ainvoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Go to https://www.bbc.com/news and fetch the first 5 news titles.",
            }
        ]
    },
    config=config,
)

result = browser_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What's the first news title?",
            }
        ]
    },
    config=config,
)
print(result["messages"][-1].content)
